# M2-B1 — Audit de qualité et effet du nettoyage

Les **outils** sont fournis (rapport de qualité, comparaison des modèles). Les **constats**, les **décisions**
et leur **interprétation** sont à toi : réponds dans les cellules ✏️.

In [ ]:
import os
import sys
from pathlib import Path

import pandas as pd
from dotenv import load_dotenv
from sqlalchemy import create_engine, text

ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
sys.path.insert(0, str(ROOT))
load_dotenv(ROOT / ".env")
engine = create_engine(os.environ["DATABASE_URL"])

df = pd.read_sql(text("""
    SELECT h.code AS hotel, r.*
    FROM reservations AS r JOIN hotels AS h ON h.hotel_id = r.hotel_id
    ORDER BY r.arrival_date
"""), engine, parse_dates=["arrival_date", "reservation_status_date"])
df.shape

## Tâche 2 — L'audit

In [ ]:
# Bornes « raisonnables » par colonne numérique : hypothèses de départ, à discuter
BORNES = {
    "lead_time": (0, 730),
    "adults": (1, 10),
    "children": (0, 10),
    "babies": (0, 5),
    "stays_in_weekend_nights": (0, 20),
    "stays_in_week_nights": (0, 50),
    "adr": (0, 1000),
}


def rapport_qualite(d: pd.DataFrame) -> pd.DataFrame:
    """Manquants et valeurs hors bornes, colonne par colonne."""
    lignes = []
    for col in d.columns:
        info = {"colonne": col, "type": str(d[col].dtype),
                "manquants": int(d[col].isna().sum()), "valeurs_distinctes": d[col].nunique()}
        if col in BORNES:
            bas, haut = BORNES[col]
            info["hors_bornes"] = int(((d[col] < bas) | (d[col] > haut)).sum())
        lignes.append(info)
    return pd.DataFrame(lignes).set_index("colonne")


rapport = rapport_qualite(df)
rapport[(rapport["manquants"] > 0) | (rapport.get("hors_bornes", 0) > 0)]

In [ ]:
# Doublons : réservations identiques sur toutes les colonnes, sauf l'identifiant
doublons = df.drop(columns="reservation_id").duplicated(keep="first")
print(doublons.sum(), "lignes en double")

### ✏️ Ce que le rapport montre

Pour chaque problème : combien de lignes, et à quoi elles ressemblent (affiche quelques exemples).
Les doublons sont-ils forcément des erreurs ? 

*…*

### ✏️ Ce que le rapport ne voit pas

Le rapport ne connaît que des bornes colonne par colonne. Cherche les **incohérences métier** : des combinaisons
de valeurs impossibles ou suspectes pour un hôtel. Au moins deux, chiffrées. Écris le code dans les cellules
ci-dessous.



In [ ]:
# TODO : tes requêtes d'investigation

## Tâche 3 — Les décisions

| Anomalie | Lignes | Décision (supprimer / corriger / imputer / garder et signaler) | Justification | Question au client ? |
|---|---|---|---|---|
| `children` manquant | | imputer à 0 (règle d'exemple) | | |
| | | | | |

## Tâche 6 — Comparer v2.0.0 et v2.1.0

À faire **après** avoir branché le nettoyage et produit la v2.1.0 avec `python train.py`.
La fonction de comparaison est fournie : c'est le **choix du jeu de test** qui t'appartient.

In [ ]:
import joblib
from sklearn.metrics import f1_score, precision_score, recall_score

import train as T
from features import INPUT_FEATURES


def comparer(modeles: dict, X: pd.DataFrame, y: pd.Series) -> pd.DataFrame:
    lignes = []
    for nom, modele in modeles.items():
        pred = modele.predict(X)
        lignes.append({"modèle": nom, "f1": f1_score(y, pred), "rappel": recall_score(y, pred),
                       "précision": precision_score(y, pred), "n": len(y)})
    return pd.DataFrame(lignes).set_index("modèle").round(3)


modeles = {
    "v2.0.0 (données brutes)": joblib.load(ROOT / "model" / "cancellation_model_v2.0.0.joblib"),
    # "v2.1.0 (données nettoyées)": joblib.load(ROOT / "model" / "cancellation_model_v2.1.0.joblib"),
}

brut = T.load_data(engine)
_, test_brut = T.split_train_test(brut)
comparer(modeles, test_brut[INPUT_FEATURES], test_brut[T.TARGET])

In [ ]:
# TODO : construire le test nettoyé, et comparer les deux modèles dessus aussi

### ✏️ Interprétation

1. Le score monte, baisse ou reste stable : pourquoi ? Une baisse est-elle forcément une mauvaise nouvelle ?
2. Quelles réservations ont été exclues, et que fera le modèle s'il en reçoit de semblables en production ?
3. Faut-il appliquer le même nettoyage aux réservations que recevra l'API ?
4. Lequel des deux jeux de test, brut ou nettoyé, ressemble le plus à la production ? Lequel retiens-tu pour décider ?

*…*